# So sánh reranker trên bộ paraphrase (Colab GPU)

Notebook này đo `hybrid+rerank` cho **luồng mô tả tự do**, tách **dev / test**:

| tập | số query | dùng để |
|---|---|---|
| dev | 500 (mẫu đã dùng để chốt luồng rerank) | chọn reranker |
| test | 2.675 (phần còn lại của 3.175 query sạch) | chạy **một lần**, lấy số báo cáo |

**Thứ tự bắt buộc:** chạy dev (mục 2), tùy chọn fine-tune (mục 3, cũng đo trên dev), **chốt model** (mục 4), rồi mới chạy test (mục 5). Đừng chạy test rồi quay lại đổi model.

**Trước khi chạy**
1. *Runtime → Change runtime type → T4 GPU*.
2. Chuẩn bị `colab/rerank_bundle.zip` (khoảng 7 MB: code + dữ liệu).
3. **Dữ liệu sẽ rời máy sang Google Colab.** Văn bản đã scrub nhưng vẫn còn tên service, mã lỗi, luồng nghiệp vụ. Kiểm tra quy định của đơn vị trước khi tải lên.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install "sentence-transformers==6.0.1" datasets accelerate

## 1. Tải bundle lên và giải nén

In [ ]:
from google.colab import files
up = files.upload()            # chon rerank_bundle.zip
BUNDLE = next(iter(up))
!unzip -oq {BUNDLE} -d /content/work
%cd /content/work
!ls pipeline data/processed

## 2. Dev: bge-reranker-base (mốc) vs bge-reranker-v2-m3

Cùng `rerank_k=30`, `max_length=256` như lần đo trước, để chênh lệch chỉ đến từ model. Lần đầu mã hóa corpus bằng e5 mất vài phút.

Kiểm tra nhanh: keyword R@10 phải là **0,576**, hybrid **0,584**, rerank bge-base khoảng **0,634** (số đo trên CPU trước đây; GPU có thể lệch rất nhỏ).

In [ ]:
!python pipeline/eval_rerankers.py --split dev --device cuda --batch-size 64 \
    --rerankers BAAI/bge-reranker-base BAAI/bge-reranker-v2-m3

Tùy chọn: v2-m3 với `max_length=512` (chậm hơn). Đây cũng là một lựa chọn làm trên dev.

In [ ]:
# !python pipeline/eval_rerankers.py --split dev --device cuda --batch-size 32 --max-length 512 \
#     --rerankers BAAI/bge-reranker-base BAAI/bge-reranker-v2-m3 --out data/processed/rerank_eval_dev_len512.json

## 3. (Tùy chọn, mặc định TẮT) Fine-tune cross-encoder

**Đọc trước khi bật:**
- `train_cross_encoder_clean.jsonl` vẫn chứa 1.452/3.175 issue của bộ đo (chủ yếu easy_neg/hard_neg). Văn bản các issue đó chứa `actual`, tức chính là query. Ô dưới **bỏ mọi dòng chạm vào key dev hoặc test**: còn 3.538/5.269 dòng.
- Cặp train là issue ↔ issue, còn bộ đo là mô tả ngắn → issue. Đây cũng là kiểu lệch nghi là nguyên nhân fine-tune e5 thua model gốc, nên đừng kỳ vọng nhiều.
- Epoch, lr cố định; không dùng dev để chọn checkpoint.

In [ ]:
RUN_FINETUNE = False

if RUN_FINETUNE:
    import json, sys, random, numpy as np, torch
    from collections import Counter
    from datasets import Dataset
    from sentence_transformers.cross_encoder import (CrossEncoder, CrossEncoderTrainer,
                                                     CrossEncoderTrainingArguments)
    from sentence_transformers.cross_encoder.losses import BinaryCrossEntropyLoss
    sys.path.insert(0, "pipeline")
    from eval_rerankers import filter_train_rows, make_split

    SEED, EPOCHS, BATCH, LR, MAX_LEN = 13, 1, 16, 2e-5, 256
    BASE_CE = "BAAI/bge-reranker-base"
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

    clean = json.load(open("data/processed/finetune_split_clean.json", encoding="utf-8"))["paraphrase_clean_keys"]
    dev = json.load(open("data/processed/paraphrase_rerank_sample500.json", encoding="utf-8"))["paraphrase_clean_keys"]
    split = make_split(clean, dev)
    rows = [json.loads(l) for l in open("data/processed/train_cross_encoder_clean.jsonl", encoding="utf-8") if l.strip()]
    kept = filter_train_rows(rows, split["dev"] + split["test"])
    print(len(rows), "->", len(kept), Counter(r["source"] for r in kept))

    ds = Dataset.from_dict({"query": [r["query"] for r in kept],
                            "doc": [r["doc"] for r in kept],
                            "label": [float(r["label"]) for r in kept]})
    model = CrossEncoder(BASE_CE, num_labels=1, max_length=MAX_LEN, device="cuda")
    args = CrossEncoderTrainingArguments(
        output_dir="ckpt-ce", num_train_epochs=EPOCHS, per_device_train_batch_size=BATCH,
        learning_rate=LR, warmup_ratio=0.1, fp16=True, seed=SEED,
        logging_steps=20, save_strategy="no", report_to="none")
    CrossEncoderTrainer(model=model, args=args, train_dataset=ds,
                        loss=BinaryCrossEntropyLoss(model)).train()
    OUT_CE = "bge-reranker-base-jira"
    model.save(OUT_CE)
    json.dump({"base": BASE_CE, "n_rows": len(kept), "epochs": EPOCHS, "batch": BATCH,
               "lr": LR, "max_len": MAX_LEN, "seed": SEED,
               "excluded": "moi dong cham key dev/test cua paraphrase"},
              open(f"{OUT_CE}/finetune_meta.json", "w"), indent=1)

In [ ]:
if RUN_FINETUNE:
    !python pipeline/eval_rerankers.py --split dev --device cuda --batch-size 64 \
        --rerankers BAAI/bge-reranker-base bge-reranker-base-jira \
        --out data/processed/rerank_eval_dev_ft.json

## 4. Chốt model trên dev

Quy tắc chốt (quyết định **trước** khi nhìn test):
- Chọn model mới chỉ khi dòng `rerank:<mới> - rerank:BAAI/bge-reranker-base` trên dev có Recall@10 **dương và CI không chứa 0**. Nếu không, giữ `BAAI/bge-reranker-base`.
- Ghi lại `pairs_per_s` trong file JSON cho phần latency.

Điền model đã chốt vào ô dưới.

In [ ]:
CHOSEN = "BAAI/bge-reranker-base"      # doi thanh model da chot, vd "BAAI/bge-reranker-v2-m3"
CHOSEN_MAX_LEN = 256

## 5. Test: chạy MỘT lần

Luôn đo mốc bge-reranker-base cùng model đã chốt, để có keyword / hybrid / rerank-base / rerank-chốt trên cùng 2.675 query. Script từ chối chạy lại nếu file kết quả đã có.

In [ ]:
RR = "BAAI/bge-reranker-base" if CHOSEN == "BAAI/bge-reranker-base" else f"BAAI/bge-reranker-base {CHOSEN}"
!python pipeline/eval_rerankers.py --split test --device cuda --batch-size 64 \
    --max-length {CHOSEN_MAX_LEN} --rerankers {RR}

## 6. Tải kết quả về

Giải nén vào `D:/viettel/semantic/` (các file nằm sẵn đường dẫn `data/processed/`), rồi báo lại để cập nhật HANDOFF.

In [ ]:
!zip -q rerank_results.zip data/processed/rerank_eval_*.json
files.download("rerank_results.zip")
# Neu da fine-tune va muon giu model (~1 GB):
# !zip -qr bge-reranker-base-jira.zip bge-reranker-base-jira
# files.download("bge-reranker-base-jira.zip")